# HMM Parameter Tuning Demo

This notebook demonstrates the interactive parameter tuning framework for HMM models.

## Features
- Interactive widget-based parameter tuning
- Real-time model training and evaluation
- Configuration saving and loading
- Results comparison across experiments
- Grid search optimization
- Bayesian optimization (optional)

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Import HMM components
import sys
sys.path.insert(0, '../py')

from imp.tuning import HMMParameterTuner, TuningConfig
from imp.tuning.optimization import (
    GridSearchOptimizer,
    quick_grid_search,
    create_default_param_grid
)

# Set random seed for reproducibility
np.random.seed(42)

print("✅ Imports successful!")

## 1. Generate Synthetic Data

Create synthetic market data with 3 distinct regimes.

In [ ]:
def generate_regime_data(n_samples=500, n_features=3):
    """
    Generate synthetic data with 3 market regimes:
    - Regime 0: Low volatility, positive trend
    - Regime 1: High volatility, mean reverting
    - Regime 2: Medium volatility, negative trend
    """
    # Regime 0: Low volatility, positive trend
    n0 = n_samples // 3
    regime0 = np.random.randn(n0, n_features) * 0.3 + np.array([0.5, 0.3, 0.4])
    
    # Regime 1: High volatility, mean reverting
    n1 = n_samples // 3
    regime1 = np.random.randn(n1, n_features) * 1.2 + np.array([0.0, 0.0, 0.0])
    
    # Regime 2: Medium volatility, negative trend
    n2 = n_samples - n0 - n1
    regime2 = np.random.randn(n2, n_features) * 0.6 + np.array([-0.4, -0.3, -0.5])
    
    # Combine and shuffle
    observations = np.vstack([regime0, regime1, regime2])
    
    # Create true labels for visualization
    true_labels = np.concatenate([
        np.zeros(n0),
        np.ones(n1),
        np.ones(n2) * 2
    ])
    
    # Shuffle together
    indices = np.random.permutation(len(observations))
    observations = observations[indices]
    true_labels = true_labels[indices]
    
    return observations, true_labels

# Generate data
observations, true_labels = generate_regime_data(n_samples=500, n_features=3)

print(f"Generated {len(observations)} observations with {observations.shape[1]} features")
print(f"Data shape: {observations.shape}")
print(f"Data range: [{observations.min():.2f}, {observations.max():.2f}]")

In [ ]:
# Visualize the generated data
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i in range(3):
    axes[i].scatter(range(len(observations)), observations[:, i], 
                   c=true_labels, cmap='viridis', alpha=0.6, s=10)
    axes[i].set_title(f'Feature {i}')
    axes[i].set_xlabel('Time')
    axes[i].set_ylabel('Value')
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 Data visualization shows 3 distinct regimes (colors)")

## 2. Interactive Parameter Tuning

Use the interactive widget interface to tune HMM parameters.

In [ ]:
# Create parameter tuner
tuner = HMMParameterTuner(
    observations=observations,
    config_dir=Path('./tuning_configs')
)

# Display the interactive interface
tuner.create_tuning_interface()

### Instructions for Interactive Tuning:

1. **Adjust Parameters**: Use the sliders and dropdowns to set:
   - Number of states (2-10)
   - Library (hmmlearn or pomegranate)
   - Covariance type (full, diag, spherical)
   - Number of iterations
   - Validation split

2. **Train Model**: Click "Train Model" to train with current parameters

3. **Review Results**: Check the metrics and visualizations

4. **Save Configuration**: Click "Save Config" to save successful configurations

5. **Compare Results**: After training multiple models, click "Compare Results"

## 3. Programmatic Parameter Tuning

You can also tune parameters programmatically without the widget interface.

In [ ]:
# Example: Train with specific configuration
from imp.hmm.trainer import EnhancedHMMTrainer

config = TuningConfig(
    n_states=3,
    library='hmmlearn',
    covariance_type='full',
    n_iterations=100,
    validation_split=0.2
)

print(f"Training with configuration: {config}")

trainer = EnhancedHMMTrainer(
    n_states=config.n_states,
    library=config.library,
    covariance_type=config.covariance_type,
    random_state=config.random_state
)

artifact, metrics = trainer.train_with_validation(
    observations,
    validation_split=config.validation_split,
    n_iterations=config.n_iterations
)

print("\n✅ Training completed!")
print(f"\nValidation Metrics:")
for metric, value in metrics.items():
    if isinstance(value, (int, float)):
        print(f"  {metric}: {value:.4f}")

## 4. Grid Search Optimization

Automatically search over a grid of parameters to find the best configuration.

In [ ]:
# Define parameter grid
param_grid = {
    'n_states': [2, 3, 4],
    'library': ['hmmlearn'],
    'covariance_type': ['full', 'diag'],
    'random_state': [42]
}

print("Starting grid search...")
print(f"Testing {3 * 1 * 2} parameter combinations\n")

# Run grid search
optimizer = GridSearchOptimizer(
    observations=observations,
    param_grid=param_grid,
    scoring_metric='log_likelihood',
    higher_is_better=True,
    n_iterations=50,  # Fewer iterations for demo
    verbose=True
)

result = optimizer.fit()

print(f"\n{'='*60}")
print("GRID SEARCH RESULTS")
print(f"{'='*60}")
print(f"Best parameters: {result.best_params}")
print(f"Best score: {result.best_score:.4f}")
print(f"Optimization time: {result.optimization_time:.2f} seconds")

In [ ]:
# Visualize grid search results
results_df = pd.DataFrame([
    {
        'n_states': r['params']['n_states'],
        'covariance_type': r['params']['covariance_type'],
        'score': r['score']
    }
    for r in result.all_results if r['score'] is not None
])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Scores by number of states
for cov_type in results_df['covariance_type'].unique():
    data = results_df[results_df['covariance_type'] == cov_type]
    axes[0].plot(data['n_states'], data['score'], 'o-', label=cov_type, markersize=8)

axes[0].set_xlabel('Number of States')
axes[0].set_ylabel('Log-Likelihood')
axes[0].set_title('Grid Search Results by State Count')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Bar chart comparison
x = range(len(results_df))
colors = ['blue' if ct == 'full' else 'orange' for ct in results_df['covariance_type']]
axes[1].bar(x, results_df['score'], color=colors, alpha=0.7)
axes[1].set_xlabel('Configuration')
axes[1].set_ylabel('Log-Likelihood')
axes[1].set_title('All Configurations')
axes[1].set_xticks(x)
axes[1].set_xticklabels([f"{r['n_states']}-{r['covariance_type'][:4]}" 
                         for _, r in results_df.iterrows()], rotation=45)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 5. Quick Grid Search

Use the convenience function for quick parameter search.

In [ ]:
# Quick grid search with default parameters
result = quick_grid_search(
    observations,
    n_states_range=[2, 3, 4],
    covariance_types=['full', 'diag'],
    verbose=True
)

print(f"\nBest configuration: {result.best_params}")
print(f"Best score: {result.best_score:.4f}")

## 6. Bayesian Optimization (Optional)

If scikit-optimize is installed, you can use Bayesian optimization for more efficient parameter search.

In [ ]:
try:
    from imp.tuning.optimization import BayesianOptimizer, quick_bayesian_search
    
    print("Running Bayesian optimization...\n")
    
    # Define parameter space
    param_space = {
        'n_states': {'type': 'integer', 'low': 2, 'high': 6},
        'library': {'type': 'categorical', 'categories': ['hmmlearn']},
        'covariance_type': {'type': 'categorical', 'categories': ['full', 'diag', 'spherical']}
    }
    
    optimizer = BayesianOptimizer(
        observations=observations,
        param_space=param_space,
        n_calls=10,  # Number of optimization iterations
        n_iterations=50,
        verbose=True
    )
    
    result = optimizer.fit()
    
    print(f"\nBest parameters: {result.best_params}")
    print(f"Best score: {result.best_score:.4f}")
    
except ImportError:
    print("⚠️ Bayesian optimization requires scikit-optimize")
    print("Install with: pip install scikit-optimize")

## 7. Accessing Tuner Results

Access and analyze results from the interactive tuner.

In [ ]:
# Get best result from tuner
if len(tuner.results) > 0:
    best_result = tuner.get_best_result(metric='log_likelihood', higher_is_better=True)
    
    if best_result:
        print("Best Result from Interactive Tuning:")
        print(f"  Experiment ID: {best_result.experiment_id}")
        print(f"  States: {best_result.config.n_states}")
        print(f"  Library: {best_result.config.library}")
        print(f"  Covariance: {best_result.config.covariance_type}")
        print(f"  Log-Likelihood: {best_result.metrics.get('log_likelihood', 'N/A')}")
else:
    print("No results yet. Train some models using the interactive interface above!")

In [ ]:
# Export all results
if len(tuner.results) > 0:
    export_path = Path('./tuning_results.json')
    tuner.export_results(export_path)
    print(f"Results exported to: {export_path}")

## 8. Summary

This notebook demonstrated:

1. ✅ **Interactive Parameter Tuning** - Widget-based interface for real-time experimentation
2. ✅ **Programmatic Tuning** - Direct API for automated workflows
3. ✅ **Grid Search** - Exhaustive search over parameter combinations
4. ✅ **Bayesian Optimization** - Efficient search using Gaussian Processes
5. ✅ **Results Management** - Save, load, and compare configurations
6. ✅ **Visualization** - Comprehensive plots for analysis

### Next Steps:

- Try different parameter ranges
- Test with real market data
- Compare hmmlearn vs pomegranate
- Integrate best models into production pipeline